# Getting Started with ContextNet

**ContextNet** extends [NicheNet](https://github.com/saeyslab/nichenetr) by replacing its generic gene regulatory network with a **context-specific GRN** constructed from your own single-cell multiomics data. The practical goal is not only to rank ligands, but to ask whether a sender-cell signal can plausibly explain a receiver-cell transcriptional program through regulatory edges that are active in the biological system being studied.

## Biological Question

In this introductory tutorial we ask: **which extracellular ligands are most consistent with the basal epithelial identity program in contralateral HNSCC tissue?** Basal epithelial cells occupy the progenitor layer of stratified epithelium, so a ligand that ranks highly here is interpreted as a candidate upstream signal for genes that distinguish basal cells from related epithelial layers.

This is a deliberately simple biological question. It teaches the core objects and interpretation without requiring replicate-aware differential analysis.

## Learning Objectives

By the end of this tutorial, you will be able to:

- Import a context-specific GRN from a gene x TF matrix
- Integrate it with NicheNet's ligand-receptor and signaling priors
- Run ligand activity prediction using cell type comparison mode
- Interpret ligand activity scores and ligand-target heatmaps
- Understand what `ContextGRN`, `ContextNetwork`, and `ContextResult` store
- Extract tables and matrices from ContextNet objects for custom analysis

## Framework Overview

The framework comprises three components:

1. **GRN inference workflow**: constructs context-specific TF-target relationships from matched scRNA-seq + scATAC-seq data.
2. **ContextNet**: integrates that GRN with NicheNet and performs ligand activity prediction.
3. **[ccflowR](https://github.com/settylab/ccflowR)**: visualizes prioritized sender-ligand-receptor-target-receiver paths as Sankey flows in the full workflows.

> **When to use this tutorial:**
> - You are new to ContextNet and want the smallest end-to-end example.
> - You want to understand the object model before running larger analyses.
> - You want to learn what can and cannot be inferred from ligand activity scores.
>
> **Use another tutorial instead if:**
> - You want full ccflowR visualization and export -> [02-standard-workflow](02-standard-workflow.ipynb)
> - You have biological replicates and multiple conditions -> [03-multicondition-workflow](03-multicondition-workflow.ipynb)

## What is a Context-Specific GRN?

A **gene regulatory network (GRN)** describes how transcription factors (TFs) regulate target genes. NicheNet uses a generic GRN compiled from public databases (TRRUST, RegNetwork, etc.). While comprehensive, these networks may not capture the regulatory relationships active in your specific experimental context.

A **context-specific GRN** is inferred directly from your single-cell multiomics data (scRNA-seq + scATAC-seq), capturing TF-target relationships that are actually active in your cells.

| Aspect | Generic GRN (NicheNet default) | Context-Specific GRN (ContextNet) |
|--------|-------------------------------|-----------------------------------|
| Source | Public databases | Your multiomics data |
| Coverage | Broad (all known interactions) | Focused (active in your context) |
| Specificity | May include irrelevant edges | Reflects your cell types/conditions |
| Result | Generic predictions | Context-relevant predictions |

## Setup

Run this notebook from the ContextNet repository root or from the `notebooks/` directory. The setup cell below normalizes the working directory to the repository root so all paths are relative and portable.

In [ ]:
suppressPackageStartupMessages({
    library(ContextNet)
    library(Seurat)
    library(nichenetr)
    library(ggplot2)
})

theme_set(theme_minimal(base_family = "sans"))

In [ ]:
check_ecosystem(verbose = TRUE)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    USER CONFIGURATION                       ║
# ║  Paths are relative to the ContextNet project root.          ║
# ╚══════════════════════════════════════════════════════════════╝

PROJECT_DIR <- normalizePath(getwd(), mustWork = TRUE)
if (basename(PROJECT_DIR) == "notebooks" &&
    file.exists(file.path(PROJECT_DIR, "..", "DESCRIPTION"))) {
    PROJECT_DIR <- normalizePath(file.path(PROJECT_DIR, ".."), mustWork = TRUE)
}
if (!file.exists(file.path(PROJECT_DIR, "DESCRIPTION"))) {
    stop("Run this notebook from the ContextNet repository root or from the notebooks/ directory.")
}
setwd(PROJECT_DIR)
cat("Working directory:", getwd(), "\n")

DATA_PATH       <- "data"
GRN_FILE        <- file.path(DATA_PATH, "gene_x_tf.csv")
SEURAT_FILE     <- file.path(DATA_PATH, "seurat_obj.rds")

# NicheNet prior knowledge networks. This directory should contain:
# - lr_network_human_21122021.rds
# - weighted_networks_nsga2r_final.rds
# - ligand_target_matrix_nsga2r_final.rds (optional but useful for comparison)
NICHENET_DB     <- DATA_PATH

# Analysis parameters: cell type comparison for basal identity
SENDER          <- NULL           # Auto-detect: all cell types with >500 cells
RECEIVER        <- "basal"
GROUP_COL       <- "Fine_Cell_Types"
IDENT_COL       <- "Fine_Cell_Types"
IDENT_1         <- "basal"
IDENT_2         <- c("parabasal", "suprabasal")
SUBSET_CATEGORY <- "C"            # Subset Seurat to contralateral tissue
MIN_CELLS       <- 500            # Minimum cells for sender cell types
TOP_N_LIGANDS   <- 20             # Number of ligands shown in the quick-start plot

stopifnot(file.exists(GRN_FILE), file.exists(SEURAT_FILE))

### NicheNet Prior Knowledge Networks

ContextNet keeps NicheNet's ligand-receptor and intracellular signaling priors, then replaces the gene-regulatory layer with your context-specific GRN. You therefore need NicheNet's prior network `.rds` files available locally before running `integrate_networks()`.

**Required inputs for this tutorial (already in `data/`):**

- `lr_network_human_21122021.rds`: ligand-receptor interactions.
- `weighted_networks_nsga2r_final.rds`: weighted signaling and regulatory prior networks.

**Optional reference (not used by the integration step):**

- `ligand_target_matrix_nsga2r_final.rds`: the original NicheNet ligand-target matrix, useful only if you want to compare ContextNet against generic NicheNet.

This is a **full local-data tutorial**: it reads the priors and a Seurat object already present under `data/` and does **not** download anything while running. The priors are distributed with the official NicheNet v2 tutorials (Zenodo record 7074291). If you are starting a new project and do not yet have them, see the optional one-time download cell below.

In [ ]:
# Verify the NicheNet prior files needed for integration are present locally.
# This tutorial does NOT download anything during execution.
required_nichenet_files <- c(
    "lr_network_human_21122021.rds",      # ligand-receptor priors
    "weighted_networks_nsga2r_final.rds"  # signaling + regulatory priors
)
# Optional reference file (not required by integrate_networks()):
optional_nichenet_files <- c("ligand_target_matrix_nsga2r_final.rds")

missing_required <- required_nichenet_files[
    !file.exists(file.path(NICHENET_DB, required_nichenet_files))
]
if (length(missing_required) > 0) {
    stop(
        "Missing required NicheNet prior files in '", NICHENET_DB, "': ",
        paste(missing_required, collapse = ", "),
        ". See the optional download cell below to fetch them once."
    )
}

cat("Required NicheNet prior files are present in", NICHENET_DB, "\n")
list.files(NICHENET_DB, pattern = "(lr_network|weighted_networks|ligand_target_matrix).*\\.rds$")

### Optional: one-time download of the NicheNet priors

If you do **not** already have the prior files, run the snippet below **once** in an R console to fetch them into your data directory. It is a setup step, not part of the executed tutorial, so it is shown as a non-executed code block.

```r
# One-time setup (run in an R console, not as part of this tutorial):
nichenet_data <- load_nichenet_data(cache_dir = NICHENET_DB, version = "v2")
```

For a persistent cache shared across projects, point `NICHENET_DB` at a directory such as `"~/.cache/nichenet"`. References: the [NicheNet step-by-step](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_steps.md) and [wrapper](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_wrapper.md) tutorials, and [Zenodo record 7074291](https://zenodo.org/record/7074291).

### Reproducibility Checkpoint

Record the exact prior files used in the analysis. File sizes and checksums are not biologically interesting, but they are useful when the same notebook is rerun months later or on another system.


In [ ]:
# Record which prior files were used (handy when rerunning months later).
nichenet_used <- c(required_nichenet_files, optional_nichenet_files)
nichenet_used <- nichenet_used[file.exists(file.path(NICHENET_DB, nichenet_used))]
nichenet_prior_paths <- file.path(NICHENET_DB, nichenet_used)

prior_file_info <- data.frame(
    file = basename(nichenet_prior_paths),
    size_mb = round(file.info(nichenet_prior_paths)$size / 1024^2, 2),
    md5 = tools::md5sum(nichenet_prior_paths),
    row.names = NULL
)
print(prior_file_info)

lr_network <- readRDS(file.path(NICHENET_DB, "lr_network_human_21122021.rds"))
if ("from" %in% colnames(lr_network) && !"ligand" %in% colnames(lr_network)) {
    lr_network$ligand <- lr_network$from
}
if ("to" %in% colnames(lr_network) && !"receptor" %in% colnames(lr_network)) {
    lr_network$receptor <- lr_network$to
}
cat("Ligand-receptor prior:", nrow(lr_network), "interactions,",
    length(unique(lr_network$ligand)), "ligands,",
    length(unique(lr_network$receptor)), "receptors\n")

## Quick Example: 4-Step Workflow

```
GRN (csv) -> import_grn() -> integrate_networks() -> predict_ligand_activity_nichenet() -> interpret results
```

This quick start is designed to answer one focused biological question while introducing the objects you will reuse in later tutorials:

> **Which ligands are plausible upstream regulators of the basal epithelial identity program in contralateral HNSCC tissue?**

We use **cell type comparison mode** (`de_subset_receiver = FALSE`). In this mode, ContextNet first identifies genes that distinguish basal cells from related epithelial layers, then asks which ligands have high regulatory potential toward those basal-enriched genes. This is useful when the goal is to interpret a cell identity or state program, rather than a disease-vs-control contrast.

The key outputs are:

- a `ContextGRN`, which tells us what regulatory edges were imported;
- a `ContextNetwork`, which stores the context-specific ligand-target matrix;
- a `ContextResult`, which stores ranked ligands, supporting targets, and the analysis parameters for this specific question.

A high-ranking ligand should be read as a **communication hypothesis**: its predicted targets overlap the basal gene program in this context-specific network. It is not by itself proof that the ligand is secreted, that the receptor binds in vivo, or that the interaction is spatially proximal.

### Step 1: Import Your GRN

The GRN is a gene x TF matrix where rows are target genes, columns are transcription factors, and values are regulatory weights inferred from matched scRNA-seq + scATAC-seq data.

After import, inspect the object before moving on. A useful GRN should have enough TFs, target genes, and non-zero edges to support network propagation. If the GRN is very sparse or uses a different gene naming convention than the expression object, ligand-target propagation will be weak or empty.

In [ ]:
grn_path <- GRN_FILE
cat("GRN file:", grn_path, "\n")

grn <- import_grn(grn_path)
grn_edges <- edges(grn)

cat("Imported ContextGRN\n")
cat("  TFs:", length(tfs(grn)), "\n")
cat("  Target genes:", length(targets(grn)), "\n")
cat("  Regulatory edges:", nrow(grn_edges), "\n")

head(grn_edges)

### GRN Quality Control

Before propagating ligand effects through the network, check whether the GRN has reasonable density and gene-symbol conventions. These checks catch common problems such as an all-zero matrix, lowercase symbols, or a GRN that contains very few target genes.


In [ ]:
gr_matrix <- as.matrix(read.csv(grn_path, row.names = 1, check.names = FALSE))
gr_nonzero <- sum(gr_matrix != 0, na.rm = TRUE)
gr_density <- gr_nonzero / (nrow(gr_matrix) * ncol(gr_matrix))

cat("Raw GRN matrix:", nrow(gr_matrix), "target genes x", ncol(gr_matrix), "TFs\n")
cat("Non-zero entries:", gr_nonzero, sprintf("(%.3f%% density)\n", 100 * gr_density))
cat("Example target genes:", paste(head(rownames(gr_matrix), 10), collapse = ", "), "\n")
cat("Example TFs:", paste(head(colnames(gr_matrix), 10), collapse = ", "), "\n")

if (gr_nonzero == 0) {
    stop("The imported GRN has no non-zero regulatory weights.")
}
if (mean(grepl("^[A-Z0-9.-]+$", rownames(gr_matrix))) < 0.5) {
    warning("Less than half of GRN target names look like uppercase HGNC-style symbols; check gene naming conventions.")
}


This output is the first quality-control checkpoint. The number of TFs, targets, and regulatory edges tells you whether the imported GRN has enough structure to support propagation. The previewed edge table shows the regulatory evidence ContextNet will use later; these rows are TF-target relationships, not ligand-target predictions yet.

### Step 2: Integrate with NicheNet

`integrate_networks()` combines three layers:

1. NicheNet ligand-receptor priors.
2. NicheNet intracellular signaling priors.
3. Your context-specific TF-target GRN.

In `mode = "replace"`, ContextNet replaces NicheNet's generic GRN with your imported GRN. The output is a `ContextNetwork` containing a ligand-target matrix: rows are target genes, columns are ligands, and values are regulatory potential scores propagated through the integrated network.

This object is reusable. Once built, the same `ContextNetwork` can be used for many sender/receiver questions on the same biological system.

In [ ]:
# This can take several minutes because it propagates signal through the integrated network.
network <- integrate_networks(
    grn,
    nichenet_database = NICHENET_DB,
    mode = "replace"
)

ltm <- ligand_target_matrix(network)
cat("Created ContextNetwork\n")
cat("  Ligands:", length(ligands(network)), "\n")
cat("  Target genes:", length(targets(network)), "\n")
cat("  Ligand-target matrix:", nrow(ltm), "targets x", ncol(ltm), "ligands\n")

integration_params(network)

This output confirms that the GRN has been converted into a reusable signaling model. The ligand-target matrix is the central `ContextNetwork` data object: each column is a candidate ligand, each row is a possible receiver target gene, and each value is a propagated regulatory potential score. The integration parameters should be reported in methods because they define how the context-specific GRN was combined with NicheNet priors.

### Step 3: Run Communication Analysis

Now we apply the reusable `ContextNetwork` to one biological question. We subset to contralateral tissue (`SUBSET_CATEGORY = "C"`) and compare basal cells against parabasal and suprabasal cells.

Because `SENDER = NULL`, the tutorial automatically uses all cell types with at least `MIN_CELLS` cells as candidate sender populations. This keeps the quick start broad: it asks which ligands from the available cellular microenvironment could explain the basal epithelial program.

For this quick start, `SENDER = NULL` deliberately uses all sufficiently abundant cell types as candidate senders. That is useful for discovery, but it is broader than a curated biological hypothesis. In a publication analysis, replace this automatic sender set with cell types that are anatomically and biologically plausible sources of the signal.


In [ ]:
seurat_obj <- readRDS(SEURAT_FILE)
seurat_c <- subset(seurat_obj, subset = category == SUBSET_CATEGORY)

celltype_counts <- sort(table(seurat_c[[GROUP_COL]][, 1]), decreasing = TRUE)
if (is.null(SENDER)) {
    SENDER <- names(celltype_counts)[celltype_counts > MIN_CELLS]
}

expr_genes <- rownames(seurat_c)
gr_targets_in_expr <- intersect(targets(grn), expr_genes)
network_targets_in_expr <- intersect(targets(network), expr_genes)
network_ligands_in_expr <- intersect(ligands(network), expr_genes)

cat("Cells in selected tissue category:", ncol(seurat_c), "\n")
cat("Candidate sender cell types with >", MIN_CELLS, "cells:\n", sep = "")
print(celltype_counts[SENDER])
cat("Receiver cell type:", RECEIVER, "\n")
cat("DE comparison:", paste(IDENT_1, collapse = "+"), "vs", paste(IDENT_2, collapse = "+"), "\n\n")

cat("Gene/network overlap QC:\n")
cat("  GRN targets in Seurat object:", length(gr_targets_in_expr), "/", length(targets(grn)), "\n")
cat("  ContextNetwork targets in Seurat object:", length(network_targets_in_expr), "/", length(targets(network)), "\n")
cat("  ContextNetwork ligands measured in Seurat object:", length(network_ligands_in_expr), "/", length(ligands(network)), "\n")

if (length(gr_targets_in_expr) < 100) {
    warning("Few GRN targets overlap the expression object; check species and gene-symbol conventions.")
}


This output defines the biological search space for the quick start. The sender list determines which cell types can contribute ligands. The receiver and comparison define the receiver-cell gene program that NicheNet will try to explain. If expected sender cell types are missing here, lower `MIN_CELLS` cautiously or inspect the cell-type labels in `GROUP_COL`.

### Receiver Program Definition

The ligand ranking is only interpretable if the receiver gene program is clear. Here we explicitly compute basal-enriched genes relative to parabasal and suprabasal cells, inspect the top markers, and pass the resulting `geneset_oi` into ContextNet.


In [ ]:
Idents(seurat_c) <- seurat_c[[IDENT_COL]][, 1]

basal_markers <- Seurat::FindMarkers(
    seurat_c,
    ident.1 = IDENT_1,
    ident.2 = IDENT_2,
    only.pos = TRUE,
    min.pct = 0.25
)
basal_markers$gene <- rownames(basal_markers)

logfc_col <- intersect(c("avg_log2FC", "avg_logFC"), colnames(basal_markers))[1]
if (is.na(logfc_col)) {
    stop("Could not find a log-fold-change column in FindMarkers output.")
}

basal_markers <- basal_markers[order(basal_markers$p_val_adj, -basal_markers[[logfc_col]]), ]
geneset_oi <- basal_markers$gene[
    basal_markers$p_val_adj < 0.05 &
        basal_markers[[logfc_col]] > 0.25
]
geneset_oi <- intersect(geneset_oi, rownames(ligand_target_matrix(network)))

cat("Basal-enriched genes retained for ligand activity:", length(geneset_oi), "\n")
cat("Geneset overlap with ContextNetwork targets:", length(geneset_oi), "genes\n")
head(basal_markers[, c("gene", logfc_col, "pct.1", "pct.2", "p_val_adj")], 15)

if (length(geneset_oi) < 20) {
    warning("The receiver gene set is small; ligand activity rankings may be unstable.")
}


In [ ]:
# Runs NicheNet ligand activity prediction using the ContextNet ligand-target matrix.
result <- predict_ligand_activity_nichenet(
    object = seurat_c,
    network = network,
    sender = SENDER,
    receiver = RECEIVER,
    group_col = GROUP_COL,
    ident_col = IDENT_COL,
    ident_1 = IDENT_1,
    ident_2 = IDENT_2,
    de_subset_receiver = FALSE,
    geneset_oi = geneset_oi,
    nichenet_database = NICHENET_DB,
    top_n_ligands = TOP_N_LIGANDS
)

activities <- ligand_activities(result)
activities <- activities[order(activities$aupr_corrected, decreasing = TRUE), ]
best_ligands <- head(activities$test_ligand, TOP_N_LIGANDS)

cat("Top ligands ranked by corrected AUPR (AUPR_c):\n")
print(head(activities[, c("test_ligand", "aupr_corrected")], TOP_N_LIGANDS))


### Sanity Checks on Top Ligands

A high activity score means that a ligand's predicted targets recover the basal gene program. It does not guarantee that the ligand is expressed by a sender population or that a cognate receptor is expressed in the receiver. The next table adds those two checks for the top-ranked ligands.


In [ ]:
result_params <- parameters(result)
cat("ContextResult stored geneset_oi:", length(result_params$geneset_oi), "genes\n")
cat("Top genes in stored receiver program:", paste(head(result_params$geneset_oi, 20), collapse = ", "), "\n\n")

expr_mat <- Seurat::GetAssayData(seurat_c, assay = DefaultAssay(seurat_c), layer = "data")
cell_groups <- seurat_c[[GROUP_COL]][, 1]
receiver_cells <- colnames(seurat_c)[cell_groups == RECEIVER]
sender_cells <- colnames(seurat_c)[cell_groups %in% SENDER]

mean_expr <- function(genes, cells) {
    genes <- intersect(genes, rownames(expr_mat))
    if (length(genes) == 0 || length(cells) == 0) return(setNames(numeric(0), genes))
    Matrix::rowMeans(expr_mat[genes, cells, drop = FALSE])
}

ligand_expr <- mean_expr(best_ligands, sender_cells)
receiver_receptors <- unique(lr_network$receptor[lr_network$ligand %in% best_ligands])
receptor_expr <- mean_expr(receiver_receptors, receiver_cells)

lr_support <- unique(lr_network[lr_network$ligand %in% best_ligands, c("ligand", "receptor")])
lr_support$sender_ligand_mean <- ligand_expr[lr_support$ligand]
lr_support$receiver_receptor_mean <- receptor_expr[lr_support$receptor]
lr_support <- lr_support[order(lr_support$ligand, -lr_support$sender_ligand_mean, -lr_support$receiver_receptor_mean), ]

head(lr_support, 30)


This ranked table is the first biological result. `test_ligand` names the candidate ligand and `aupr_corrected` is the corrected ligand activity score (`AUPR_c`). Higher values mean better recovery of the basal-enriched target gene set. Treat this as a prioritized hypothesis list, then use the heatmap and ligand-target table to identify which genes support each ligand.

### Step 4: Visualize Results

The first plot ranks ligands by corrected AUPR (`AUPR_c`). This is the primary ligand activity score used for ordering. The y-axis is ordered from lower to higher rank so the strongest ligand appears at the top.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)

activities <- ligand_activities(result)
activities <- activities[order(activities$aupr_corrected, decreasing = TRUE), ]
top_act <- head(activities, TOP_N_LIGANDS)
top_act$test_ligand <- factor(top_act$test_ligand, levels = rev(top_act$test_ligand))

best_ligands <- as.character(top_act$test_ligand)

ggplot(top_act, aes(x = aupr_corrected, y = test_ligand)) +
    geom_col(fill = "#2C7BB6", width = 0.75) +
    labs(
        x = "Corrected ligand activity (AUPR_c)",
        y = NULL,
        title = "Ligands Prioritized for the Basal Epithelial Program",
        subtitle = "Ranked by corrected AUPR from ContextNet/NicheNet ligand activity prediction"
    ) +
    theme_minimal(base_family = "sans") +
    theme(plot.title = element_text(face = "bold"))

**Interpretation**: Each bar shows corrected AUPR (`AUPR_c`), which summarizes how well that ligand's predicted target genes recover the basal-enriched receiver program. The plot should be read as a ranking of candidate upstream signals. It does not show ligand expression by itself, and it does not prove direct cell-cell contact.

The second plot asks a more mechanistic question: **which target genes connect the top ligands to the basal program?** We subset the context-specific ligand-target matrix to the top ligands and top predicted targets. Darker tiles indicate stronger propagated regulatory potential.

`nichenetr::make_heatmap_ggplot()` requires a base R `matrix`. In some ContextNet installations, `ligand_target_matrix(network)` can be represented as a sparse matrix-like object after subsetting, so the plotting subset is explicitly converted with `as.matrix()` before plotting.

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 10)

best_targets <- top_targets(result, n = 50)
ltm <- ligand_target_matrix(network)

target_rows <- intersect(best_targets, rownames(ltm))
ligand_cols <- intersect(best_ligands, colnames(ltm))
vis_matrix <- as.matrix(ltm[target_rows, ligand_cols, drop = FALSE])

nonzero <- vis_matrix[vis_matrix > 0]
if (length(nonzero) > 0) {
    vis_matrix[vis_matrix < stats::quantile(nonzero, 0.33)] <- 0
}
vis_matrix <- vis_matrix[rowSums(vis_matrix) > 0, , drop = FALSE]

cat("Heatmap matrix:", nrow(vis_matrix), "targets x", ncol(vis_matrix), "ligands\n")

if (nrow(vis_matrix) == 0 || ncol(vis_matrix) == 0) {
    stop("No ligand-target links remain for plotting. Try lowering the filtering threshold or increasing TOP_N_LIGANDS.")
}

nichenetr::make_heatmap_ggplot(
    vis_matrix,
    y_name = "Predicted target gene",
    x_name = "Prioritized ligand",
    legend_title = "Regulatory\npotential",
    color = "purple",
    x_axis_position = "top"
)

This heatmap converts the ligand ranking into a mechanism-oriented view. Rows are predicted receiver target genes and columns are prioritized ligands. The color scale is regulatory potential from the ContextNet ligand-target matrix after light filtering for readability. Dense columns suggest broad regulatory programs; isolated tiles suggest more specific ligand-target hypotheses.

### Path-Level Visualization with ccflowR

The bar plot ranks ligands and the heatmap shows ligand-target support. ccflowR adds the path-level view: **which sender cell types, ligands, receptors, and basal target programs form the strongest communication hypotheses?**

Read each path as `sender -> ligand -> receptor -> target program -> basal receiver`. Wider flows represent higher ContextNet-supported scores after expression and receptor support are folded into the visualization data. The plot is a prioritization device for follow-up biology, not direct proof of secretion, receptor binding, or spatial contact.


In [ ]:
# Build path-level Sankey data from the result, then scale + threshold it.
sankey_raw <- prepare_for_ccflowr(
    result,
    expression_data = seurat_c,
    network = network,
    group_col = GROUP_COL,
    top_n_ligands = TOP_N_LIGANDS
)
sankey_data <- combine_sankey_data(sankey_raw, threshold_quantile = 0.95)

cat("ccflowR paths retained:", nrow(sankey_data), "\n")
head(sankey_data)

In [ ]:
if (requireNamespace("ccflowR", quietly = TRUE)) {
    cell_types_in_plot <- unique(c(sankey_data$sender, sankey_data$receiver))
    palette <- grDevices::hcl.colors(length(cell_types_in_plot), palette = "Dark 3")
    cell_colors <- stats::setNames(palette, cell_types_in_plot)
    ccflowR::sankey_flow(sankey_data, cell_colors)
} else {
    message("ccflowR is not installed. Install it to render the Sankey graph: remotes::install_github('settylab/ccflowR')")
}


Use the Sankey graph to choose a small number of interpretable paths for validation. For example, a wide fibroblast-to-basal path suggests a sender population and ligand-receptor pair to check against ligand expression, receptor expression in basal cells, spatial proximity, and perturbation evidence.


## Exploring ContextNet Objects

ContextNet uses S4 objects so that each stage of the analysis keeps both data and provenance. The object classes are intentionally narrow:

| Object | Created by | Biological meaning | Main data inside |
|--------|------------|--------------------|------------------|
| `ContextGRN` | `import_grn()` | TF-target regulatory edges active in the profiled context | edge table, TFs, targets, regulons |
| `ContextNetwork` | `integrate_networks()` | NicheNet prior with the context-specific GRN embedded | ligand-target matrix and integration parameters |
| `ContextResult` | `predict_ligand_activity_nichenet()` or `predict_ligand_activity_multinichenet()` | ranked candidate signals for the tested sender/receiver question | ligand activities, top ligands, targets, ligand-target links, parameters |

This separation matters biologically. `ContextGRN` is the regulatory evidence from your multiomic data; `ContextNetwork` is the propagated signaling model; `ContextResult` is the answer to one specific biological contrast.

### Getting Data Out of ContextNet Objects

Use accessor functions rather than reaching into S4 slots. Accessors give stable, analysis-ready tables and matrices.

#### `ContextGRN`

| Accessor | Returns | Typical use |
|----------|---------|-------------|
| `edges(grn)` | Edge list with regulator, target, and weight columns | audit TF-target evidence, filter weak edges |
| `tfs(grn)` | TF names | check TF coverage |
| `targets(grn)` | target gene names | check target-gene overlap with expression data |
| `regulons(grn)` | named list of TF -> targets | inspect a TF regulon |
| `length(grn)` | number of GRN edges | quick complexity check |

#### `ContextNetwork`

| Accessor | Returns | Typical use |
|----------|---------|-------------|
| `ligand_target_matrix(network)` | matrix of target genes x ligands | plug into nichenetr or custom scoring |
| `ligands(network)` | ligand names | check candidate-ligand coverage |
| `targets(network)` | target gene names | check target coverage after integration |
| `integration_params(network)` | mode, damping factor, network versions | report reproducible analysis settings |

#### `ContextResult`

| Accessor | Returns | Typical use |
|----------|---------|-------------|
| `ligand_activities(result)` | ranked ligand activity table | primary result table |
| `top_ligands(result)` | top-ranked ligands | plotting and focused validation |
| `top_targets(result)` | top predicted target genes | downstream enrichment or marker checks |
| `ligand_target_links(result)` | ligand-target links for top predictions | mechanistic interpretation |
| `parameters(result)` | sender, receiver, contrast, thresholds | methods reporting |
| `as.data.frame(result)` | ligand activity table | generic export and tidy workflows |
| `provenance(result)` | creation metadata | reproducibility audit |
| `metadata(result)` | user/package metadata | additional annotations |

All objects track **provenance** automatically, including creation time, package version, and source function.

In [ ]:
# Pull analysis-ready data out of ContextNet objects.
grn_edges <- edges(grn)
ltm <- ligand_target_matrix(network)
activities_tbl <- ligand_activities(result)
links_tbl <- ligand_target_links(result)
params_tbl <- parameters(result)

cat("ContextGRN edge table:", nrow(grn_edges), "rows\n")
cat("ContextNetwork ligand-target matrix:", nrow(ltm), "targets x", ncol(ltm), "ligands\n")
cat("ContextResult ligand activity table:", nrow(activities_tbl), "ligands\n")
cat("ContextResult ligand-target link table:", nrow(links_tbl), "links\n")

head(activities_tbl[order(activities_tbl$aupr_corrected, decreasing = TRUE), ], 5)

This final extraction cell shows the standard way to move from tutorial inspection to downstream analysis. The accessors return ordinary R objects: data frames for edges and ligand activities, a matrix for ligand-target regulatory potential, and a parameter list for reproducibility. Use these extracted objects for custom visualization, enrichment analysis, reporting tables, or direct calls into `nichenetr`.

The extracted objects are ordinary R objects: data frames, character vectors, lists, and matrices. This is the intended route for downstream work such as enrichment analysis, custom plotting, comparison to perturbation screens, or direct use of the context-specific matrix in `nichenetr`.

In [ ]:
sessionInfo()

## Key Parameters

| Parameter | Required | Description |
|-----------|----------|-------------|
| `sender` | Yes | Sender cell type(s) expressing ligands |
| `receiver` | Yes | Receiver cell type expressing receptors and targets |
| `ident_col` | Yes* | Metadata column for DE grouping |
| `ident_1` | Yes* | Value(s) for DE group 1 (upregulated genes) — can be a vector |
| `ident_2` | Yes* | Value(s) for DE group 2 (reference) — can be a vector |
| `de_subset_receiver` | No | Whether to subset to receiver cells first (default: TRUE) |
| `geneset_oi` | No | Custom gene set; auto-computed from DE if NULL |
| `group_col` | No | Metadata column with cell type labels (default: "cell_type") |

*Required unless `geneset_oi` is provided directly

### Two DE Comparison Modes

**Mode 1: Condition comparison** (`de_subset_receiver = TRUE`, default)

Compare conditions *within* the receiver cell type. Use when you have experimental conditions (disease/control, treated/untreated).

```r
# Compare Disease vs Control WITHIN Fibroblast cells
result <- predict_ligand_activity_nichenet(
    object = seurat_obj, network = network,
    sender = "Macrophage", receiver = "Fibroblast",
    ident_col = "disease_status",
    ident_1 = "Disease", ident_2 = "Control"
)
```

**Mode 2: Cell type comparison** (`de_subset_receiver = FALSE`)

Compare cell types directly *across* all cells. Use when finding cell type-specific marker genes.

```r
# Find basal-specific genes vs parabasal/suprabasal
result <- predict_ligand_activity_nichenet(
    object = seurat_obj, network = network,
    sender = all_celltypes, receiver = "basal",
    ident_col = "cell_type",
    ident_1 = "basal", ident_2 = c("parabasal", "suprabasal"),
    de_subset_receiver = FALSE
)
```

## Next Steps

- **[02-standard-workflow](02-standard-workflow.ipynb)**: Full NicheNet workflow with condition comparison, Sankey visualization, and export
- **[03-multicondition-workflow](03-multicondition-workflow.ipynb)**: Multi-condition differential analysis with biological replicates (MultiNicheNet)

### Using nichenetr / multinichenetr directly

ContextNet's main contribution is the **context-specific ligand-target matrix**. If you prefer to run your own analysis code rather than using ContextNet's wrapper functions, extract the matrix after Step 2 and follow the upstream tutorials:

```r
# Extract the context-specific ligand-target matrix
ltm <- ligand_target_matrix(network)

# Use directly in nichenetr or multinichenetr workflows
# — replace the default ligand_target_matrix with ltm
```

- **nichenetr tutorials**: [Step-by-step analysis](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_steps.md) | [Seurat wrapper](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_wrapper.md) | [Circos visualization](https://github.com/saeyslab/nichenetr/blob/master/vignettes/seurat_wrapper_circos.md)
- **multinichenetr tutorials**: [Basic analysis](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/basic_analysis_steps_MISC.md) | [Pairwise comparison](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/pairwise_analysis_MISC.md) | [Paired design](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/paired_analysis_SCC.knit.md)